# Patrones de ventas + narrativa analítica — Superstore

- **Bloque 1** — detectar patrones: categorías dominantes, consistencia por región, atípicos
- **Bloque 2** — convertir hallazgos en insights accionables


## Setup

In [1]:
import pandas as pd
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT = find_project_root()
df   = pd.read_csv(ROOT / 'data' / 'external' / 'train.csv')
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')
print(f'Shape: {df.shape}')

Shape: (9800, 18)


---
# Bloque 1 — Patrones de ventas


## 1.1 — Ventas por categoría


In [2]:
cat = (
    df.groupby('Category')['Sales']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'Sales': 'revenue'})
)

# Proporción de cada categoría sobre el total
cat['pct'] = (cat['revenue'] / cat['revenue'].sum() * 100).round(1)
cat['revenue'] = cat['revenue'].round(0)

print(cat.to_string(index=False))
print(f'\nTotal revenue: ${cat["revenue"].sum():,.0f}')

       Category  revenue  pct
     Technology 827456.0 36.6
      Furniture 728659.0 32.2
Office Supplies 705422.0 31.2

Total revenue: $2,261,537


## 1.2 — Top 5 subcategorías


In [3]:
subcat = (
    df.groupby('Sub-Category')['Sales']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'Sales': 'revenue'})
    .head(5)
)

subcat['pct'] = (subcat['revenue'] / df['Sales'].sum() * 100).round(1)
subcat['revenue'] = subcat['revenue'].round(0)

print('Top 5 subcategorías:')
print(subcat.to_string(index=False))

top5_pct = subcat['pct'].sum()
print(f'\nLas 5 primeras subcategorías concentran el {top5_pct}% del revenue total')

Top 5 subcategorías:
Sub-Category  revenue  pct
      Phones 327782.0 14.5
      Chairs 322823.0 14.3
     Storage 219343.0  9.7
      Tables 202811.0  9.0
     Binders 200029.0  8.8

Las 5 primeras subcategorías concentran el 56.3% del revenue total


## 1.3 — ¿El patrón se mantiene por región?

Pivot `Region × Category` para comparar el orden de dominancia por zona.


In [4]:
# pivot: filas = Region, columnas = Category, valores = suma de Sales
pivot = (
    df.groupby(['Region', 'Category'])['Sales']
    .sum()
    .unstack()
    .round(0)
)

print(pivot)

# Categoría dominante en cada región
dominante_por_region = pivot.idxmax(axis=1)
print('\nCategoría dominante por región:')
print(dominante_por_region)

Category  Furniture  Office Supplies  Technology
Region                                          
Central    160317.0         163590.0    168739.0
East       206461.0         199941.0    263117.0
South      116531.0         124425.0    148195.0
West       245348.0         217467.0    247405.0

Categoría dominante por región:
Region
Central    Technology
East       Technology
South      Technology
West       Technology
dtype: str


## 1.4 — Comportamientos atípicos

Regiones donde la categoría dominante difiere de la dominante global.


In [5]:
dominante_global = cat.iloc[0]['Category']
print(f'Dominante global: {dominante_global}')

for region, categ in dominante_por_region.items():
    if categ != dominante_global:
        rev_atip = pivot.loc[region, categ]
        rev_glob = pivot.loc[region, dominante_global]
        diferencia = round((rev_atip - rev_glob) / rev_glob * 100, 1)
        print(f'\n[ATÍPICO] {region}: domina {categ} (${rev_atip:,.0f})')
        print(f'          vs {dominante_global}: ${rev_glob:,.0f} ({diferencia:+.1f}%)')

if all(v == dominante_global for v in dominante_por_region.values):
    print('Sin atípicos: el patrón global se repite en todas las regiones')

Dominante global: Technology
Sin atípicos: el patrón global se repite en todas las regiones


---
# Bloque 2 — Narrativa analítica


## La cadena del insight

| Paso | Pregunta | Ejemplo |
|------|----------|--------|
| **Observación** | ¿Qué ves en los datos? | "Technology genera $827K" |
| **Patrón** | ¿Se repite? ¿En qué segmentos? | "Lidera en 3 de 4 regiones" |
| **Interpretación** | ¿Qué significa para el negocio? | "Revenue depende de productos de precio alto" |
| **Implicación** | ¿Qué acción sugiere? | "Una caída en tech afecta el total de forma no lineal" |

Un insight sin implicación es un dato con palabras encima.


## Estructura de un informe de hallazgos

```
Contexto:     ¿Sobre qué dataset? ¿Qué período? ¿Qué se analizó?
Hallazgo N:   [nombre del patrón]
              Observación → Patrón → Interpretación → Implicación
Recomendación: Acción concreta para el stakeholder
```


## Mini-informe — 3 hallazgos accionables

Los datos reales de la sección anterior se traducen aquí a lenguaje de negocio.
Se ejecuta el código para actualizar los números y se lee la narrativa debajo.


In [6]:
# Recoge los números calculados en el Bloque 1 para construir la narrativa
cat1      = cat.iloc[0]
cat3      = cat.iloc[2]
top5_pct  = subcat['pct'].sum()
top1_sub  = subcat.iloc[0]['Sub-Category']
top1_pct  = subcat.iloc[0]['pct']

atipicos = [
    (region, categ)
    for region, categ in dominante_por_region.items()
    if categ != dominante_global
]

print('=== MINI-INFORME: Patrones de ventas Superstore ===')
print(f'Dataset: train.csv · {len(df):,} transacciones')
print()
print('HALLAZGO 1 — Concentración de revenue en una categoría')
print(f'  Obs:    {cat1["Category"]} genera ${cat1["revenue"]:,.0f} ({cat1["pct"]}% del total)')
print(f'  Patrón: lidera en todas las regiones donde es dominante')
print(f'  Interp: el revenue depende de un segmento de productos de ticket alto')
print(f'  Implic: una contracción en demanda tecnológica reduce el total de forma no lineal')
print()
print('HALLAZGO 2 — El 80/20 a nivel de subcategoría')
print(f'  Obs:    las 5 primeras subcategorías concentran el {top5_pct}% del revenue')
print(f'  Patrón: {top1_sub} sola representa el {top1_pct}% del total')
print(f'  Interp: la distribución de ventas no es uniforme — hay una cabeza muy concentrada')
print(f'  Implic: priorizar stock y visibilidad de esas 5 subcategorías tiene impacto desproporcionado')
print()
print('HALLAZGO 3 — Comportamientos atípicos por región')
if atipicos:
    for region, categ in atipicos:
        print(f'  Obs:    en {region} domina {categ}, no {dominante_global}')
        print(f'  Patrón: todas las demás regiones siguen el orden global')
        print(f'  Interp: {region} tiene un perfil de cliente o mercado diferente')
        print(f'  Implic: las estrategias comerciales de {region} deben diferenciarse del plan global')
else:
    print(f'  El patrón global se replica en todas las regiones — no hay atípicos geográficos')

=== MINI-INFORME: Patrones de ventas Superstore ===
Dataset: train.csv · 9,800 transacciones

HALLAZGO 1 — Concentración de revenue en una categoría
  Obs:    Technology genera $827,456 (36.6% del total)
  Patrón: lidera en todas las regiones donde es dominante
  Interp: el revenue depende de un segmento de productos de ticket alto
  Implic: una contracción en demanda tecnológica reduce el total de forma no lineal

HALLAZGO 2 — El 80/20 a nivel de subcategoría
  Obs:    las 5 primeras subcategorías concentran el 56.3% del revenue
  Patrón: Phones sola representa el 14.5% del total
  Interp: la distribución de ventas no es uniforme — hay una cabeza muy concentrada
  Implic: priorizar stock y visibilidad de esas 5 subcategorías tiene impacto desproporcionado

HALLAZGO 3 — Comportamientos atípicos por región
  El patrón global se replica en todas las regiones — no hay atípicos geográficos


---
## Checklist de un buen insight

Antes de presentar un hallazgo, validar:

- [ ] ¿Hay un número concreto que lo soporte?
- [ ] ¿Se menciona el contexto (qué dataset, qué período)?
- [ ] ¿Se identifica si es un patrón repetido o un dato aislado?
- [ ] ¿La interpretación va más allá de describir el número?
- [ ] ¿Hay una implicación o recomendación para el negocio?
- [ ] ¿Puede leerlo alguien sin conocimiento técnico?

Si falta alguno, el insight no está terminado.
